In [ ]:
import urllib.request, zipfile
url = ("https://archive.ics.uci.edu/static/public/366/"
       "activity+recognition+system+based+on+multisensor+data+fusion+arem.zip")
urllib.request.urlretrieve(url, "arem.zip")
zipfile.ZipFile("arem.zip").extractall("data")

# Time Series Classification Part 1: Feature Creation/Extraction

## 1(a)–(b): Loading and Splitting the Data

According to the assignment instructions, we partition the activity dataset files using the following rules:
* **For `bending1` and `bending2` folders**: Datasets 1 and 2 are kept as **Test data**. All other datasets are **Train data**.
* **For all other activity folders**: Datasets 1, 2, and 3 are kept as **Test data**. All other datasets are **Train data**.

In [ ]:
import os
import glob
import pandas as pd

# The folder where your zipfile extracted the files
root_dir = "data" 

# One metadata record per instance (train and test together)
instances = []

# List out the activity directories present in the data folder
folders = ['bending1', 'bending2', 'cycling', 'lying',
           'sitting', 'standing', 'walking']
for folder in folders:
    folder_path = os.path.join(root_dir, folder)
    
    # Ensure the folder exists before scanning
    assert os.path.exists(folder_path), f"Missing folder: {folder_path}"
        
    # Gather all CSV files within the activity folder
    # Sort numerically so dataset10 comes after dataset9, not after dataset1
    files = sorted(
        glob.glob(os.path.join(folder_path, "dataset*.csv")),
        key=lambda p: int("".join(filter(str.isdigit, os.path.basename(p)))))
    
    for file_path in files:
        file_name = os.path.basename(file_path)
        
        # Dataset number from the filename, e.g. 'dataset3.csv' -> 3
        dataset_num = int(''.join(filter(str.isdigit, file_name)))
            
        # Split rule from 1(b)
        if folder in ['bending1', 'bending2']:
            is_test = dataset_num in [1, 2]
        else:
            is_test = dataset_num in [1, 2, 3]
            
        # Store metadata and file path mapping
        file_info = {
            "activity": folder,
            "file_name": file_name,
            "file_path": file_path,
            "dataset_num": dataset_num,
            "split": "test" if is_test else "train"
        }
        
        instances.append(file_info)

meta = pd.DataFrame(instances)
print(meta["split"].value_counts())
assert len(meta) == 88
assert (meta["split"] == "train").sum() == 69
assert (meta["split"] == "test").sum() == 19


## Loading the Time Series and Verifying Shapes

Each instance file contains **6 time series**: `avg_rss12`, `var_rss12`, `avg_rss13`, `var_rss13`, `avg_rss23`, and `var_rss23`, each with **480 consecutive values** (plus a `time` column, which we drop).

The files begin with `#` comment lines, and a few use spaces instead of commas or have trailing delimiters. The loader below parses each line directly so all of these cases are handled, and it checks the shape of every file.

In [ ]:
import re

cols = ["time", "avg_rss12", "var_rss12", "avg_rss13",
        "var_rss13", "avg_rss23", "var_rss23"]

def load_timeseries_csv(path):
    """Read one AReM file: skip '#' lines, handle comma/space
    separators and trailing delimiters, drop the time column."""
    rows = []
    with open(path) as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            # Drop empty strings left by trailing delimiters
            vals = [v for v in re.split(r"[,\s]+", line) if v]
            assert len(vals) == 7, \
                f"{path}, line {line_no}: {len(vals)} fields -> {line!r}"
            rows.append([float(v) for v in vals])
    df = pd.DataFrame(rows, columns=cols)
    return df.drop(columns="time")

# Load all 88, aligned with rows of meta
series = [load_timeseries_csv(p) for p in meta["file_path"]]

# Verify every file
bad = [(p, s.shape) for p, s in zip(meta["file_path"], series)
       if s.shape != (480, 6)]
print(f"Loaded {len(series)} files; {len(bad)} with unexpected shape")
for p, shape in bad:
    print(p, shape)

display(series[0].head())

### **Conclusion on Data Loading & Verification**

All 88 files loaded. `sitting/dataset8.csv` contains 479 rows instead of 480 (one missing observation in the source file). It is kept as-is, since the time-domain features are computed over whatever values are present and are not meaningfully affected.

# 1(c) Feature Extraction

## 1(c)i. Research on Time-Domain Features in Time Series Classification

Common time-domain features used to summarize time series data include:

* **Statistical moments:** mean (central level), variance / standard deviation (spread), skewness (asymmetry), and kurtosis (tail heaviness, i.e. how prone the series is to outliers).
* **Extreme values and ranges:** minimum, maximum, absolute maximum, and peak-to-peak range (max − min).
* **Percentiles and quantiles:** median, 1st quartile (25th percentile), 3rd quartile (75th percentile), and interquartile range (IQR = Q3 − Q1), which are robust to outliers.
* **Temporal dynamics:** autocorrelation coefficients (self-similarity across lags), root mean square (RMS, overall signal magnitude), zero-crossing rate (how often the signal crosses its mean), mean absolute change (average step-to-step variation), slope (overall trend), number of peaks, and time-domain energy (sum of squared values).

For 1(c)ii, we use the seven features specified in the assignment: minimum, maximum, mean, median, standard deviation, 1st quartile, and 3rd quartile.

## 1(c)ii. Extracting the Time-Domain Features

For each of the 88 instances, we compute 7 statistics (min, max, mean, median, standard deviation, 1st quartile, 3rd quartile) for each of the 6 time series, giving **42 features per instance**.

In [ ]:
import numpy as np

feature_cols = ["avg_rss12", "var_rss12", "avg_rss13",
                "var_rss13", "avg_rss23", "var_rss23"]

stats = {
    "min":       lambda x: x.min(),
    "max":       lambda x: x.max(),
    "mean":      lambda x: x.mean(),
    "median":    lambda x: x.median(),
    "std":       lambda x: x.std(),          # sample std (ddof=1)
    "1st_quart": lambda x: x.quantile(0.25),
    "3rd_quart": lambda x: x.quantile(0.75),
}

rows = []
# series[k] is aligned with row k of meta
for s in series:
    row = {}
    for i, col in enumerate(feature_cols, 1):   # series 1..6
        for name, f in stats.items():
            row[f"{name}{i}"] = f(s[col])
    rows.append(row)

features = pd.DataFrame(rows)
features.index = pd.RangeIndex(1, 89, name="Instance")
features["activity"] = meta["activity"].values
features["split"] = meta["split"].values

# 42 features + activity + split
assert features.shape == (88, 44), features.shape
display(features.head())

**Result (1(c)ii):** The feature matrix has **88 rows × 44 columns**.

- **Rows:** one per instance (a single 480-step recording of one activity), indexed 1 to 88. All instances are included here; the train/test split is used in Homework 4.
- **Columns:** 7 statistics × 6 time series = **42 features**, named `<stat><i>` (e.g. `min1`, `3rd_quart6`), plus `activity` and `split`, which are kept for labeling and Homework 4.

| Index *i* | Time series |
|---|---|
| 1 | avg_rss12 |
| 2 | var_rss12 |
| 3 | avg_rss13 |
| 4 | var_rss13 |
| 5 | avg_rss23 |
| 6 | var_rss23 |

**Notes:**
- Standard deviation uses the sample formula (ddof = 1). Quartiles use pandas' default linear interpolation.
- Features are left unnormalized.
- `sitting/dataset8.csv` has 479 rows instead of 480; its features are computed over the 479 available values.

## 1(c)iii. Standard Deviation of Each Feature and 90% Bootstrap Confidence Intervals

We compute the standard deviation of each of the 42 features across all 88 instances, then use a **90% percentile bootstrap confidence interval** (resampling instances with replacement) to quantify the uncertainty in each standard deviation estimate.

In [ ]:
# Select only the 42 extracted numeric feature column headers
numeric_features = [col for col in features.columns
                    if col not in ["activity", "split"]]

n_iterations = 1000
confidence_level = 0.90  # 90% Bootstrap confidence interval
lower_pct = ((1.0 - confidence_level) / 2.0) * 100
upper_pct = (confidence_level + (1.0 - confidence_level) / 2.0) * 100

bootstrap_results = []

rng = np.random.default_rng(552)   # fixed seed for reproducibility

for col in numeric_features:
    vals = features[col].values
    
    # Point estimate: std of this feature across all instances
    point_est_sd = np.std(vals, ddof=1)
    
    # Perform bootstrapping
    boot_stds = []
    for _ in range(n_iterations):
        boot_sample = rng.choice(vals, size=len(vals), replace=True)
        boot_stds.append(np.std(boot_sample, ddof=1))
        
    ci_lower = np.percentile(boot_stds, lower_pct)
    ci_upper = np.percentile(boot_stds, upper_pct)
    
    bootstrap_results.append({
        "Feature": col,
        "SD Point Estimate": point_est_sd,
        "90% CI Lower": ci_lower,
        "90% CI Upper": ci_upper
    })

bootstrap_df = pd.DataFrame(bootstrap_results).set_index("Feature")
display(bootstrap_df)


In [ ]:
# Constant features (std exactly 0)
display(bootstrap_df[bootstrap_df["SD Point Estimate"] == 0])

# Near-constant feature
display(bootstrap_df.loc[["min6"]])

# Five most variable features
display(bootstrap_df.sort_values("SD Point Estimate", ascending=False).head(5))

**Result (1(c)iii):** For each of the 42 features, we computed the sample standard deviation across all 88 instances and a 90% percentile bootstrap confidence interval (1,000 resamples of instances with replacement, seed 552). Every point estimate lies inside its interval. Values below are rounded to 3 decimals.

- **Most variable features:**

| Feature | Std | 90% CI |
|---|---|---|
| min1 | 9.570 | [8.235, 10.689] |
| 1st_quart1 | 6.154 | [5.594, 6.658] |
| min5 | 6.124 | [4.490, 7.585] |
| 1st_quart5 | 6.096 | [4.833, 7.279] |
| median5 | 5.814 | [4.435, 6.855] |

  All five come from the average-signal series avg_rss12 (1) and avg_rss23 (5), and mostly from their lower-end statistics (min, 1st quartile). These features vary most from one instance to another, so they are candidates for separating activities.

- **Constant features:** `min2` and `min4` have std 0 with CI [0, 0]. The minimum of var_rss12 and var_rss13 is 0 in every instance, so these features carry no information for classification.

- **Near-constant feature:** `min6` has std 0.046 with CI [0.000, 0.078]. It is 0 in almost every instance, and the lower bound of 0 means many bootstrap resamples contained only zeros, so it is also of little use.

## 1(c)iv. Selection of the Three Most Important Features

We select **min, max, and mean** as the three most important time-domain features, applied to every time series.

- **Min:** the most variable feature type in 1(c)iii. `min1` has the largest standard deviation of all 42 features (9.570, 90% CI [8.235, 10.689]), and `min5` ranks third (6.124). High variability across instances makes a feature a strong candidate for separating activities.
- **Mean:** summarizes the overall signal level of a series, which differs between static activities (lying, sitting, standing) and dynamic ones (walking, cycling). Its spread across instances is also substantial (e.g. `mean1` std 5.336).
- **Max:** captures the peak signal level and complements min, so together they describe the range each series covers.

**Caveat:** min is constant for the variance series var_rss12 and var_rss13 (`min2`, `min4` have std 0), so it is informative mainly for the average-signal series. High variability across instances suggests, but does not guarantee, that a feature separates classes; we will check this with the scatter plots in Homework 4.

# 2. ISLR Exercise 3.7.4

We have n = 100 observations and fit two models: a linear regression, Y = β0 + β1·X + ε, and a cubic regression, Y = β0 + β1·X + β2·X² + β3·X³ + ε.

### (a) True relationship is linear: training RSS
We expect the **cubic model to have lower (or equal) training RSS**. The cubic model contains the linear model as a special case (β2 = β3 = 0), and least squares minimizes RSS over all coefficient choices. So the cubic fit can never do worse than the linear fit on the training data. In practice it will be strictly lower, because the extra terms bend the curve to fit some of the random noise ε.

### (b) True relationship is linear: test RSS
We expect the **linear model to have lower test RSS**. Since the truth is linear, the linear model already has essentially no bias. The extra cubic terms add no real signal; they only fit noise in the training set (overfitting), which increases variance and makes predictions on new data worse.

### (c) True relationship is nonlinear: training RSS
We expect the **cubic model to have lower training RSS**. The same nesting argument as in (a) applies, and it does not depend on the true relationship: the more flexible model always fits the training data at least as well. Here the cubic model can also capture some of the real curvature, so its advantage should be even larger.

### (d) True relationship is nonlinear: test RSS
**There is not enough information to tell.** It depends on how far the true relationship is from linear. If it is strongly nonlinear, the cubic model's lower bias should outweigh its extra variance, giving lower test RSS. If it is only slightly nonlinear, the linear model's lower variance may win. This is the bias-variance tradeoff.